In [22]:
import pandas as pd
import numpy as np 
import networkx as nx
import plotly.express as px
import dash_cytoscape as cyto
import matplotlib.pyplot as plt
import dash
from dash import Dash, html, dcc
from dash.dependencies import Output, Input, State
import dash_cytoscape as cyto

%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)


In [23]:
df = pd.read_csv('../data/processed/pac_vendor_agg.csv')

#random_100 = df.committee_name.sample(100).tolist()

df = df[(df['is_union_pac']==False) & (df['pct_unitem']>=.25)]

#df[['committee_name','pct_unitem','n_unique_vendors','is_union_pac']].drop_duplicates()


In [29]:
cyto.load_extra_layouts()  # enables advanced layouts like cose, dagre

df = pd.read_csv('../data/processed/pac_vendor_agg.csv')

# cmte_nm_filter = [
# "SUPPORT AMERICA'S POLICE PAC",
# "UNITED VETERANS ALLIANCE OF AMERICA PAC",
# "SUPPORT OUR FIREFIGHTERS AND PARAMEDICS PAC",
# "AMERICAN POLICE AND TROOPERS COALITION PAC",
# "POLICE COALITION OF AMERICA PAC",
# "AMERICAN VETERANS SUPPORT GROUP PAC",
# "POLICE AND TROOPER SUPPORT PAC",
# "CONSTITUTIONAL LEADERSHIP PAC",
# "UNITED BREAST CANCER SUPPORT PAC",
# "FIREFIGHTERS COALITION OF AMERICA PAC",
# "AMERICAN VETERANS INITIATIVE PAC",
# "VETERANS AID PAC",
# "AUTISM HEAR US NOW",
# "LAW ENFORCEMENT FOR A SAFER AMERICA PAC",
# "AMERICAN ALLIANCE FOR DISABLED CHILDREN PAC",
# "AMERICAN COALITION FOR CRISIS RELIEF PAC",
# "HONORING AMERICAN LAW ENFORCEMENT PAC",
# "AMERICAN HEALTH COMMITTEE PAC"]

# df = df[df['committee_name'].isin(cmte_nm_filter)]

#random_100 = df.committee_name.sample(100).tolist()

#df = df[df['committee_name'].isin(random_100)]

df = df[(df['is_union_pac']==False) & (df['pct_unitem']>=.25)]


def filter_data(df, percent_value, flag_selected, numeric_value, categories_selected, committees_selected):
    
    filtered_df = df.copy()

    # Slider filter
    filtered_df = filtered_df[filtered_df["pct_unitem"] >= percent_value]

    # Toggle filter
    if "TRUE" in flag_selected:
        filtered_df = filtered_df[filtered_df["is_union_pac"] == False]

    # Numeric input
    filtered_df = filtered_df[filtered_df["n_unique_vendors"] <= numeric_value]

    # Dropdown filter
    if categories_selected:
        filtered_df = filtered_df[filtered_df["committee_designation_full"].isin(categories_selected)]
    
    # Dropdown filter
    if committees_selected:
        filtered_df = filtered_df[filtered_df["committee_name"].isin(committees_selected)]
    
    filtered_df = filtered_df.sort_values(by = ['committee_id','ttl_spend'], ascending = [True,  False])

    return filtered_df


def update_viz(filtered_df):

    cmte_unique = (filtered_df[['committee_id','committee_name','committee_state','committee_type_full','committee_designation_full','organization_type_full','treasurer_name','individual_unitemized_contributions','receipts','pct_indiv','pct_unitem','n_unique_vendors','is_union_pac']].copy()
               .drop_duplicates()

    )

    vendor_unique = (filtered_df[['vendor_full','exp_cat','n_unique_filers']].copy()
                .drop_duplicates()

    )

    cmtes = [tuple(row) for row in cmte_unique.itertuples(index = False)]


    vendors = [tuple(row) for row in vendor_unique.itertuples(index = False)]


    ## Edges - CMTE -> Vendors 

    edges = [
        (row.committee_name, row.vendor_full, float(row.pct_ttl_spend))
        for _, row in filtered_df.iterrows()
    ]

    G = nx.Graph()
    G.add_nodes_from(cmte_unique['committee_name'], type = 'Filer')
    G.add_nodes_from(vendor_unique['vendor_full'], type = 'Vendor')
    G.add_weighted_edges_from(edges)

## Nodes = CMTES and VENDORS, 
### IF CMTE attributes = pct unitem, n vendors, cmte type, organization type is union, state, filer cmte id, url 
### IF VENDORS attributes = exp type, n unique filers 

 
    # Set Node Attributes 
    committee_id_dict = {}
    committee_state_dict = {}
    committee_type_dict = {}
    committee_designation_dict = {}
    organization_type_dict = {}
    treasurer_name_dict = {}
    individual_unitemized_contributions_dict = {}
    receipts_dict = {}
    pct_indiv_dict = {}
    pct_unitem_dict = {}
    n_unique_vendors_dict = {}
    is_union_pac_dict = {}

    for cmte in cmtes:
        committee_id_dict[cmte[1]] = cmte[0]
        committee_state_dict[cmte[1]] = cmte[2]
        committee_type_dict[cmte[1]] = cmte[3]
        committee_designation_dict[cmte[1]] = cmte[4]
        organization_type_dict[cmte[1]] = cmte[5]
        treasurer_name_dict[cmte[1]] = cmte[6]
        individual_unitemized_contributions_dict[cmte[1]] = cmte[7]
        receipts_dict[cmte[1]] = cmte[8]
        pct_indiv_dict[cmte[1]] = cmte[9]
        pct_unitem_dict[cmte[1]] = cmte[10]
        n_unique_vendors_dict[cmte[1]] = cmte[11]
        is_union_pac_dict[cmte[1]] = cmte[12]


    vendor_exp_cat_dict = {}
    vendor_unique_filers_dict = {}

    for vendor in vendors:
        vendor_exp_cat_dict[vendor[0]] = vendor[1] 
        vendor_unique_filers_dict[vendor[0]] = vendor[2]    


    nx.set_node_attributes(G, committee_id_dict, 'committee_id')
    nx.set_node_attributes(G, committee_state_dict, 'committee_state')
    nx.set_node_attributes(G, committee_type_dict, 'committee_type') 
    nx.set_node_attributes(G, committee_designation_dict, 'committee_designation')
    nx.set_node_attributes(G, organization_type_dict, 'organization_type')
    nx.set_node_attributes(G, treasurer_name_dict, 'treasurer_name')
    nx.set_node_attributes(G, individual_unitemized_contributions_dict, 'unitemized_contributions')
    nx.set_node_attributes(G, receipts_dict, 'receipts')
    nx.set_node_attributes(G, pct_indiv_dict, 'pct_receipts_from_individuals')
    nx.set_node_attributes(G, pct_unitem_dict, 'pct_receipts_from_unitemized')
    nx.set_node_attributes(G, n_unique_vendors_dict, 'n_unique_vendors')
    nx.set_node_attributes(G, is_union_pac_dict, 'is_union_pac')


    nx.set_node_attributes(G, vendor_exp_cat_dict, 'exp_cat')
    nx.set_node_attributes(G, vendor_unique_filers_dict, 'n_unique_filers')


    #convert to cytoscape:
    cyto_data = nx.cytoscape_data(G)
    elements = cyto_data['elements'] ['nodes'] + cyto_data['elements']['edges']
    
    return elements

app = Dash(__name__)

heading = html.H1("Federal PAC Spending Network")
subheading = html.H2("2023 - 2024 Election Cycle")


side_bar = html.Div(
        style={
            "width": "320px",
            "minWidth": "320px",
            "flexShrink": 0,
            "padding": "20px",
            "background-color": "#f8f9fa",
            "border-right": "1px solid #ddd",
            "overflow-y": "auto",
            'height': '100vh'
        }, 
        children = [

            html.H3("Filters", style={"marginBottom": "20px"}),

            # Slider
            html.Label("Minimum % of Receipts from Unitemized Contributions"),
            dcc.Slider(
                id = 'percent-slider',
                min = .25,
                max = 1.00,
                step = .05,
                marks = {round(i/4 + 0.25, 2): f"{int((i/4 + 0.25)*100)}%" for i in range(4)},
                value = .80
            ),
            html.Br(),

            # Toggle switch
            html.Label(""),
            dcc.Checklist(
                id="flag-toggle",
                options=[{"label": "Exclude Union, Trade, and Membership PACs?", "value": "TRUE"}],
                value = []
            ),
            html.Br(),

            # Numeric input
            html.Label("Maximum Number of Vendors"),
            dcc.Input(
                id="numeric-input",
                type="number",
                value=100,
                style={"width": "100%"}
            ),
            html.Br(), html.Br(),

            # Multi-select dropdown
            html.Label("Select PAC designations"),
            dcc.Dropdown(
                id="category-dropdown",
                options=[{"label": c, "value": c} for c in df["committee_designation_full"].dropna().unique()],
                multi=True,
                placeholder= "Select a PAC designation"
            ),
            # Multi-select dropdown
            html.Label("Select Filing Committee"),
            dcc.Dropdown(
                id="cmte-dropdown",
                options=[{"label": c, "value": c} for c in df["committee_name"].dropna().unique()],
                multi=True,
                placeholder= "Select a PAC"
            ),
        ], className= 'col-2'
    )

main_viz = html.Div([
        cyto.Cytoscape(
            id = 'network',
            minZoom = 0.2,
            maxZoom = 4,
            elements = [],
            style={'width': '100%', 'height': '1000px'},
            layout={'name': 'cose'}, 
            selectedNodeData = None,
            stylesheet=[]
            )
        ], className= 'col', style = {'height':'100vh', 'overflow':'hidden',
            'flex': "1"}
    )

app.layout = html.Div(
        children = [heading,
                    subheading, 
                    html.Div(
                    style={
                        "display": "flex",
                        "flexDirection": "row",
                        "height": "100vh",       
                        "overflow": "hidden"     
                    },
                    children = [
                        
                        side_bar, 
                        main_viz
                        ], className='row'
                )
        ]
)
@app.callback(Output(component_id='network',component_property='elements'),
              [Input('percent-slider', 'value'),
                Input('flag-toggle', 'value'),
                Input('numeric-input', 'value'),
                Input('category-dropdown', 'value'),
                Input('cmte-dropdown', 'value')]
              )

def update_elements(percent_value, flag_selected, numeric_value, categories_selected, committees_selected):
    filtered_df = filter_data(df, percent_value, flag_selected, numeric_value, categories_selected, committees_selected)
    elements = update_viz(filtered_df)
    return elements



@app.callback(Output(component_id='network',component_property='stylesheet'),
              [Input(component_id='network', component_property='selectedNodeData'),
               Input(component_id='network',component_property='elements')])

def update_edge_color(snd, elements):
    # Base stylesheet (normal colors)
    base_styles = [
            {
                'selector': 'edge',
                'style': {
                    'line-color': '#B8B19D',
                    'width': .25,
                    'line-opacity': .25
                }
            },
            {'selector': 'node',
                'style': {
                    'label': 'data(name)'
                }

            },

            {
                'selector': 'node[type = "Filer"]',
                'style': {
                    'width': 10,
                    'height': 10,
                    'background-color': '#384056',
                    'label': 'data(name)',
                    'font-size': 3,
                    'font-weight': 'bold',
                    'text-color': 'rgb(255,0,0)'
                }
            },
            {
                'selector': 'node[name = "INDEPENDENT EXPENDITURE"]',
                'style': {
                    'width': 7,
                    'height': 7,
                    'background-color': '#EDAA1C',
                    'label': 'data(name)',
                    'font-size': 3,
                    'font-weight': 'bold',
                    'text-color': 'rgb(255,0,0)'
                }
            },
            {
                'selector':'[exp_cat = "Other"]',
                'style':{
                    'width': 3.5,
                    'height': 3.5,
                    'background-color': '#265436',
                    'label': 'data(name)',
                    'font-size': 2
                }
            },
            {
                'selector':'[exp_cat = "Transfer to affiliated committee"]',
                'style':{
                    'width': 3.5,
                    'height': 3.5,
                    'background-color': '#799A84',
                    'label': 'data(name)',
                    'font-size': 2
                }
            },
            {
                'selector':'[exp_cat = "Contribution to Candidate or Committee"]',
                'style':{
                    'width': 3.5,
                    'height': 3.5,
                    'background-color': '#BADDC9',
                    'label': 'data(name)',
                    'font-size': 2
                }
            }
        ]
    
    if not snd:
        return base_styles

    selected_node = snd[0]['id']

    # Add new dynamic style rules
    highlight_styles = [
        # highlight edges connected to selected node
        {
            'selector': f'edge[source = "{selected_node}"], edge[target = "{selected_node}"]',
            'style': {
                'line-color': '#5D6770',
                'width': 'data(weight)',
            }
        },
        # de-emphasize other edges
        {
            'selector': f'edge[source != "{selected_node}"][target != "{selected_node}"]',
            'style': {
                'line-color': '#DDD',
                'width': .25,
                'opacity': 0.25
            }
        }
    ]

    return base_styles + highlight_styles


if __name__ == '__main__':
    app.run(debug=False)